In [ ]:
import os
import numpy as np
from sklearn.cluster import KMeans
from langchain_nebius.embeddings import NebiusEmbeddings

# ==========================================
# STEP 1: Setup and Mock Data
# ==========================================
# Imagine these are the chunks you extracted using Docling's Hybrid Chunker
docling_chunks = [
    "The basic subscription costs $10 per month and includes 500 API calls.",
    "Premium tier is $50 per month with unlimited API calls and priority support.",
    "To reset your password, navigate to the settings menu and click 'Security'.",
    "If you forget your password, the recovery email will be sent within 5 minutes.",
    "Enterprise plans require a custom contract and include dedicated account managers.",
]

# We use Chroma's default embedding function (all-MiniLM-L6-v2) for demonstration
# In production, you might use OpenAIEmbeddingFunction or Cohere
emb_fn = NebiusEmbeddings(api_key=os.environ.get("EMBEDDING_API_KEY"))
raw_embeddings = emb_fn.embed_documents(docling_chunks)
embeddings_array = np.array(raw_embeddings)

# ==========================================
# STEP 2: Spherical K-Means Clustering
# ==========================================
# CRITICAL: Normalize embeddings so K-Means (Euclidean) acts like Cosine Similarity
norms = np.linalg.norm(embeddings_array, axis=1, keepdims=True)
normalized_embeddings = embeddings_array / norms

# Define how many buckets (clusters) we want.
# For 5 chunks, 2 clusters makes sense. For 10,000 chunks, you might want 500.
num_clusters = 3

# Initialize and fit the K-Means algorithm
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init="auto")
kmeans.fit(normalized_embeddings)

# Extract the assigned bucket IDs for each chunk
# This will be an array like: [0, 0, 1, 1, 0]
cluster_assignments = kmeans.labels_
print(cluster_assignments)

[2 0 1 1 0]


: 

In [5]:
## FOR IMPORTING MODULES FROM /app

import sys
import os

# Get the directory where the notebook is running
project_root = os.path.abspath(
    os.path.join(os.getcwd(), "..")
)  # Adjust '..' based on depth

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [6]:
from typing import Any

from ragas import experiment, Dataset
from ragas.backends import LocalCSVBackend
from pydantic import BaseModel


class DatasetModelSingleTurn(BaseModel):
    id: int
    query: str
    expected_answer: str
    metadata: dict[str, Any] | None = None


backend = LocalCSVBackend(root_dir="./dataset")
dataset_1 = Dataset(
    name="dataset_1", backend=backend, data_model=DatasetModelSingleTurn
)

data = [
    DatasetModelSingleTurn(
        id=1, query="What is the capital of France?", expected_answer="Paris"
    ),
    DatasetModelSingleTurn(
        id=2,
        query="What is the chemical symbol for gold?",
        expected_answer="Au",
        metadata={"category": "science", "difficulty": "easy"},
    ),
    DatasetModelSingleTurn(
        id=3,
        query="Who wrote 'To Kill a Mockingbird'?",
        expected_answer="Harper Lee",
        metadata=None,  # Explicitly setting to None, though it's the default
    ),
    DatasetModelSingleTurn(
        id=4,
        query="What is the largest planet in our solar system?",
        expected_answer="Jupiter",
        metadata={"category": "astronomy"},
    ),
    DatasetModelSingleTurn(
        id=5,
        query="In what year did the Apollo 11 moon landing occur?",
        expected_answer="1969",
    ),
    DatasetModelSingleTurn(
        id=6,
        query="What is the speed of light in a vacuum?",
        expected_answer="Approximately 299,792 kilometers per second",
        metadata={"category": "physics", "exact_value": 299792458},
    ),
]

for d in data:
    dataset_1.append(d)

dataset_1.to_pandas()

,id,query,expected_answer,metadata
0,1,What is the capital of France?,Paris,None
1,2,What is the chemical symbol for gold?,Au,"{'category': 'science', 'difficulty': 'easy'}"
2,3,Who wrote 'To Kill a Mockingbird'?,Harper Lee,None
3,4,What is the largest planet in our solar system?,Jupiter,{'category': 'astronomy'}
4,5,In what year did the Apollo 11 moon landing oc...,1969,None
5,6,What is the speed of light in a vacuum?,"Approximately 299,792 kilometers per second","{'category': 'physics', 'exact_value': 299792458}"


In [7]:
# import asyncio

# from ragas import experiment, Dataset
# from ragas.backends import LocalCSVBackend
# from app.routes.dependencies.llm import get_chat_model_service
# from app.routes.dependencies.settings import get_app_settings
# from langchain_core.messages import HumanMessage

# app_settings = get_app_settings()
# chat_model_service = get_chat_model_service(app_settings)


# @experiment()
# async def experiment_model_prompt(
#     row: DatasetModelSingleTurn, model_name: str, temperature: float
# ):
#     model = chat_model_service.client
#     response = model.invoke([HumanMessage(content=row.query)]).content

#     return {
#         **row.model_dump(),
#         "response": response,
#         "experiment_name": f"{model_name}_temp_{temperature}",
#         "model_name": model_name,
#         "temperature": temperature,
#     }


# results_1 = await experiment_model_prompt.arun(
#     dataset=dataset_1, model_name="gpt40", temperature=1.0
# )
# results_1.to_pandas()


Running experiment:   0%|          | 0/6 [00:00<?, ?it/s]

[2026-03-06 09:29:42,662][INFO] factory:client:46             Selected model: ChatNebius
[2026-03-06 09:29:45,837][INFO] _client:_send_single_request:1025             HTTP Request: POST https://api.studio.nebius.ai/v1/chat/completions "HTTP/1.1 200 OK"
[2026-03-06 09:29:47,631][INFO] _client:_send_single_request:1025             HTTP Request: POST https://api.studio.nebius.ai/v1/chat/completions "HTTP/1.1 200 OK"
[2026-03-06 09:29:49,925][INFO] _client:_send_single_request:1025             HTTP Request: POST https://api.studio.nebius.ai/v1/chat/completions "HTTP/1.1 200 OK"
[2026-03-06 09:29:51,868][INFO] _client:_send_single_request:1025             HTTP Request: POST https://api.studio.nebius.ai/v1/chat/completions "HTTP/1.1 200 OK"
[2026-03-06 09:29:53,395][INFO] _client:_send_single_request:1025             HTTP Request: POST https://api.studio.nebius.ai/v1/chat/completions "HTTP/1.1 200 OK"
[2026-03-06 09:29:54,694][INFO] _client:_send_single_request:1025             HTTP Request:

,id,query,expected_answer,metadata,response,experiment_name,model_name,temperature
0,5,In what year did the Apollo 11 moon landing oc...,1969,None,The Apollo 11 moon landing **occurred in 1969*...,gpt40_temp_1.0,gpt40,1.0
1,4,What is the largest planet in our solar system?,Jupiter,{'category': 'astronomy'},That’s easy — the largest planet in our solar ...,gpt40_temp_1.0,gpt40,1.0
2,6,What is the speed of light in a vacuum?,"Approximately 299,792 kilometers per second","{'category': 'physics', 'exact_value': 299792458}",The speed of light in a **vacuum** is a fundam...,gpt40_temp_1.0,gpt40,1.0
3,3,Who wrote 'To Kill a Mockingbird'?,Harper Lee,None,**Harper Lee** is the author of ***To Kill a M...,gpt40_temp_1.0,gpt40,1.0
4,1,What is the capital of France?,Paris,None,The capital of France is **Paris**. Known for ...,gpt40_temp_1.0,gpt40,1.0
5,2,What is the chemical symbol for gold?,Au,"{'category': 'science', 'difficulty': 'easy'}",The chemical symbol for **gold** is **Au**. \n,gpt40_temp_1.0,gpt40,1.0


In [10]:
import os
import pandas as pd
from dotenv import load_dotenv

# LangChain Imports
from langchain_community.document_loaders import DirectoryLoader
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Ragas Imports
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.testset.transforms import (
    HeadlinesExtractor,
    HeadlineSplitter,
    KeyphrasesExtractor,
    apply_transforms,
)
from ragas.testset.persona import Persona
from ragas.testset.synthesizers.single_hop.specific import (
    SingleHopSpecificQuerySynthesizer,
)
from ragas.testset import TestsetGenerator

from app.routes.dependencies.embedding import get_embedding

app_settings = get_app_settings()
chat_model_service = get_chat_model_service(app_settings)
embedding_service = get_embedding(app_settings)


def generate_golden_dataset():
    # 1. Setup Environment & Models
    # Using a high-capability model for generation ensures high-quality golden answers
    load_dotenv()
    os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

    eval_llm = chat_model_service.client
    eval_embeddings = embedding_service.client

    # Wrap models for Ragas compatibility
    generator_llm = LangchainLLMWrapper(eval_llm)
    generator_embeddings = LangchainEmbeddingsWrapper(eval_embeddings)

    # 2. Load Source Documents
    print("Loading documents...")
    loader = DirectoryLoader(
        "./data", glob="**/*.pdf", use_multithreading=True
    )  # Adjust path and glob as needed
    docs = loader.load()

    # 3. Create the Base Knowledge Graph
    print("Initializing Knowledge Graph...")
    kg = KnowledgeGraph()
    for doc in docs:
        kg.nodes.append(
            Node(
                type=NodeType.DOCUMENT,
                properties={
                    "page_content": doc.page_content,
                    "document_metadata": doc.metadata,
                },
            )
        )

    # 4. Enrich the Knowledge Graph via Transforms
    # This extracts semantic properties so the synthesizers can build realistic queries
    print("Applying Knowledge Graph Transformations...")
    transforms = [
        HeadlinesExtractor(llm=generator_llm, max_num=10),
        HeadlineSplitter(max_tokens=1500),
        KeyphrasesExtractor(llm=generator_llm),
    ]
    apply_transforms(kg, transforms=transforms)

    # PROD TIP: Save the KG state. If generation fails or you want to generate
    # more questions later, you won't have to pay to re-compute the graph.
    kg.save("enriched_knowledge_graph.json")
    print("Knowledge Graph saved locally.")

    # 5. Define Personas
    # Personas simulate different user behaviors to ensure dataset diversity
    personas = [
        Persona(
            name="Domain Expert",
            role_description="Asks highly technical, detailed questions seeking deep, granular understanding.",
        ),
        Persona(
            name="Novice User",
            role_description="Asks simple, high-level questions and needs broad, accessible explanations.",
        ),
    ]

    # 6. Configure Synthesizers & Query Distribution
    # Map the extracted graph properties (headlines, keyphrases) to specific query generators
    query_distribution = [
        (
            SingleHopSpecificQuerySynthesizer(
                llm=generator_llm, property_name="keyphrases"
            ),
            0.6,
        ),  # 60% based on key themes
        (
            SingleHopSpecificQuerySynthesizer(
                llm=generator_llm, property_name="headlines"
            ),
            0.4,
        ),  # 40% based on section headers
    ]

    # 7. Generate the Testset
    print("Generating Golden Dataset...")
    generator = TestsetGenerator(
        llm=generator_llm,
        embedding_model=generator_embeddings,
        knowledge_graph=kg,
        persona_list=personas,
    )

    # Generate the actual Q&A pairs
    testset = generator.generate(testset_size=50, query_distribution=query_distribution)

    # 8. Export Data
    df = testset.to_pandas()
    df.to_csv("golden_dataset.csv", index=False)
    print("Golden dataset successfully generated and saved to 'golden_dataset.csv'!")

    return df


if __name__ == "__main__":
    golden_df = generate_golden_dataset()
    print(golden_df.head())

C:\Users\kairu\AppData\Local\Temp\ipykernel_26700\660659701.py:42: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(eval_llm)
C:\Users\kairu\AppData\Local\Temp\ipykernel_26700\660659701.py:43: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(eval_embeddings)


Loading documents...


[2026-03-06 09:33:08,930][INFO] __init__:<module>:13             pikepdf C++ to Python logger bridge initialized


Initializing Knowledge Graph...
Applying Knowledge Graph Transformations...


Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s][2026-03-06 09:33:34,517][INFO] _client:_send_single_request:1740             HTTP Request: POST https://api.studio.nebius.ai/v1/chat/completions "HTTP/1.1 200 OK"
[2026-03-06 09:33:40,036][INFO] _client:_send_single_request:1740             HTTP Request: POST https://api.studio.nebius.ai/v1/chat/completions "HTTP/1.1 200 OK"
Applying KeyphrasesExtractor:   0%|          | 0/48 [00:00<?, ?it/s][2026-03-06 09:33:46,017][INFO] _client:_send_single_request:1740             HTTP Request: POST https://api.studio.nebius.ai/v1/chat/completions "HTTP/1.1 200 OK"
[2026-03-06 09:33:46,082][INFO] _client:_send_single_request:1740             HTTP Request: POST https://api.studio.nebius.ai/v1/chat/completions "HTTP/1.1 200 OK"
Applying KeyphrasesExtractor:   2%|▏         | 1/48 [00:03<02:38,  3.36s/it][2026-03-06 09:33:47,247][INFO] _client:_send_single_request:1740             HTTP Request: POST https://api.studio.nebius.ai/v1/chat/

Knowledge Graph saved locally.
Generating Golden Dataset...


Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s][2026-03-06 09:34:47,158][INFO] _client:_send_single_request:1740             HTTP Request: POST https://api.studio.nebius.ai/v1/chat/completions "HTTP/1.1 200 OK"
[2026-03-06 09:34:49,547][INFO] _client:_send_single_request:1740             HTTP Request: POST https://api.studio.nebius.ai/v1/chat/completions "HTTP/1.1 200 OK"
Generating Scenarios:  50%|█████     | 1/2 [00:05<00:05,  5.88s/it][2026-03-06 09:34:50,701][INFO] _client:_send_single_request:1740             HTTP Request: POST https://api.studio.nebius.ai/v1/chat/completions "HTTP/1.1 200 OK"
[2026-03-06 09:34:53,424][INFO] _client:_send_single_request:1740             HTTP Request: POST https://api.studio.nebius.ai/v1/chat/completions "HTTP/1.1 200 OK"
[2026-03-06 09:34:56,103][INFO] _client:_send_single_request:1740             HTTP Request: POST https://api.studio.nebius.ai/v1/chat/completions "HTTP/1.1 200 OK"
[2026-03-06 09:34:58,963][INFO] _client:_send_single_r

Golden dataset successfully generated and saved to 'golden_dataset.csv'!
                                          user_input  \
0  What is the Andromeda Galaxy and how far away ...   
1                What's the Milky Way's name origin?   
2                  What are some well-known galaxys?   
3                 What is a supermassive black hole?   
4                                        star counts   

                                  reference_contexts  \
0  [3/2/26, 11:26 AM\n\nGalaxy - Wikipedia\n\nGal...   
1  [Etymology\n\nThe word galaxy was borrowed via...   
2  [Nomenclature\n\nMillions of galaxies have bee...   
3  [Milky Way galaxy that contains the Solar Syst...   
4  [sphere of the fixed stars."[29] Actual proof ...   

                                           reference   persona_name  \
0  The Andromeda Galaxy is the nearest large neig...    Novice User   
1  The word galaxy was borrowed from the Greek te...  Domain Expert   
2  Well-known galaxies include the Andro